In [56]:
import pandas as pd

In [57]:
df = pd.read_csv("globalterrorismdb_0718dist.tar.bz2", compression="bz2")

In [58]:
print(f"Shape: {df.shape}")
print(f"Year range: {df['iyear'].min()} - {df['iyear'].max()}")

Shape: (181691, 136)
Year range: 1970 - 2017


In [59]:
# selected columns

selected_cols = [
    # Date & Location
    'iyear', 'imonth', 'iday',
    'country_txt', 'region_txt', 'city',

    # Attack info
    'success', 'suicide',
    'attacktype1', 'attacktype1_txt',

    # Target info
    'targtype1_txt', 'targsubtype1_txt', 'target1', 'natlty1_txt',

    # Group info
    'gname', 'gsubname',

    # Perpetrators & Weapons
    'nperps',
    'weaptype1_txt', 'weapsubtype1_txt',

    # Casualties
    'nkill', 'nkillus'
]

df = df[selected_cols].copy()
print(f"✔ Selected {len(selected_cols)} columns")


✔ Selected 21 columns


In [60]:
# missing value
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
mv_filteACCENT = mv[mv['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print(mv_filtered.to_string())

                  Missing Count  Missing %
gsubname                 175801      96.76
nperps                    71115      39.14
nkillus                   64446      35.47
weapsubtype1_txt          20768      11.43
targsubtype1_txt          10373       5.71
nkill                     10313       5.68
natlty1_txt                1559       0.86
target1                     638       0.35
city                        435       0.24


In [61]:
# drop data gsubname cuz 96.76 % missing
df = df.drop(columns=['gsubname'])
print(f"✔ Dropped 'gsubname' column")
print(f"  Shape after dropping: {df.shape[0]:,} rows × {df.shape[1]} columns")


✔ Dropped 'gsubname' column
  Shape after dropping: 181,691 rows × 20 columns


In [62]:
# fill missing values - categorical value

cat_cols = ['city', 'targsubtype1_txt', 'target1', 'natlty1_txt', 'weapsubtype1_txt']

for col in cat_cols:
    n_missing = df[col].isnull().sum()
    df[col] = df[col].fillna('Unknown')
    print(f"  ✔ '{col}': filled {n_missing:,} missing values with 'Unknown'")

  ✔ 'city': filled 435 missing values with 'Unknown'
  ✔ 'targsubtype1_txt': filled 10,373 missing values with 'Unknown'
  ✔ 'target1': filled 638 missing values with 'Unknown'
  ✔ 'natlty1_txt': filled 1,559 missing values with 'Unknown'
  ✔ 'weapsubtype1_txt': filled 20,768 missing values with 'Unknown'


In [63]:
# fill missing values - numerical value
# nperps: -1 = GTD convention for unknown/unreported
n_missing_nperps = df['nperps'].isnull().sum()
df['nperps'] = df['nperps'].fillna(-1).astype(int)

# nkill, nkillus: 0 = no reported fatalities
n_missing_nkill = df['nkill'].isnull().sum()
n_missing_nkillus = df['nkillus'].isnull().sum()
df['nkill'] = df['nkill'].fillna(0).astype(int)
df['nkillus'] = df['nkillus'].fillna(0).astype(int)

# success & suicide: binary, fill with 0
df['success'] = df['success'].fillna(0).astype(int)
df['suicide'] = df['suicide'].fillna(0).astype(int)

In [64]:
# fix data types

df['iyear']  = df['iyear'].astype(int)
df['imonth'] = df['imonth'].astype(int)
df['iday']   = df['iday'].astype(int)
df['attacktype1'] = df['attacktype1'].astype(int)

print(df.dtypes)


iyear               int64
imonth              int64
iday                int64
country_txt           str
region_txt            str
city                  str
success             int64
suicide             int64
attacktype1         int64
attacktype1_txt       str
targtype1_txt         str
targsubtype1_txt      str
target1               str
natlty1_txt           str
gname                 str
nperps              int64
weaptype1_txt         str
weapsubtype1_txt      str
nkill               int64
nkillus             int64
dtype: object


In [65]:
# remove duplicate rows

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"✔ Removed {before - after:,} duplicate rows")
print(df.shape)

✔ Removed 11,261 duplicate rows
(170430, 20)


In [66]:
# result validation

print(f"  Final shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Year range       : {df['iyear'].min()} - {df['iyear'].max()}")
print(f"  Total missing    : {df.isnull().sum().sum()}")
print(f"  Duplicate rows   : {df.duplicated().sum()}")

  Final shape      : 170,430 rows × 20 columns
  Year range       : 1970 - 2017
  Total missing    : 0
  Duplicate rows   : 0


In [67]:
# save cleaned dataset

output_path = 'gtd_cleaned_1970_2017.csv'
df.to_csv(output_path, index=False)


# assignment

Q1 - trend kasus setiap tahun, and korelasi setiap region dengan trend global

In [68]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

In [69]:
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#141414',
    'axes.edgecolor':   '#2a2a2a',
    'axes.labelcolor':  '#aaaaaa',
    'xtick.color':      '#555555',
    'ytick.color':      '#555555',
    'text.color':       '#cccccc',
    'grid.color':       '#1e1e1e',
    'grid.linewidth':   0.8,
    'font.family':      'monospace',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
 
ACCENT   = '#e63946'   
BLUE     = '#5b9cf6'
ORANGE   = '#f4a261'
TEAL     = '#2a9d8f'
PURPLE   = '#c77dff'
YELLOW   = '#ffd166'


import matplotlib.cm as cm

def make_color_map(series, cmap_name='tab20'):
    unique_vals = sorted(series.dropna().unique())
    cmap        = cm.get_cmap(cmap_name, len(unique_vals))
    return {val: cmap(i) for i, val in enumerate(unique_vals)}

REGION_COLORS = make_color_map(df['region_txt'])
METHOD_COLORS = make_color_map(df['attacktype1_txt'], cmap_name='tab10')

In [70]:
# total global attack

global_trend = (df.groupby('iyear')
                  .size()
                  .reset_index(name='attacks'))
peak_year = global_trend.loc[global_trend['attacks'].idxmax(), 'iyear']
min_year  = global_trend.loc[global_trend['attacks'].idxmin(), 'iyear']
mean_attacks = global_trend['attacks'].mean()
print(f"\n  Peak year : {peak_year}"
      f"  ({global_trend['attacks'].max():,} attacks)")
print(f"  Min  year : {min_year}"
      f"  ({global_trend['attacks'].min():,} attacks)")
print(f"  Mean/year : {mean_attacks:,.0f}")


  Peak year : 2014  (16,109 attacks)
  Min  year : 1971  (460 attacks)
  Mean/year : 3,626


In [71]:
# region attack per year

regional_trend = (df.groupby(['iyear', 'region_txt'])
                    .size()
                    .reset_index(name='attacks'))
 
region_totals = (df.groupby('region_txt')
                   .size()
                   .reset_index(name='total')
                   .sort_values('total', ascending=False))
 
print("\nTotal attacks by region (all years):")
print(region_totals.to_string(index=False))
 
TOP6 = region_totals['region_txt'].head(6).tolist()
print(f"\nTop-6 regions selected: {TOP6}")


Total attacks by region (all years):
                 region_txt  total
 Middle East & North Africa  47765
                 South Asia  43134
         Sub-Saharan Africa  16933
              South America  16575
             Western Europe  15146
             Southeast Asia  11994
Central America & Caribbean   9037
             Eastern Europe   4959
              North America   3347
                  East Asia    714
               Central Asia    551
      Australasia & Oceania    275

Top-6 regions selected: ['Middle East & North Africa', 'South Asia', 'Sub-Saharan Africa', 'South America', 'Western Europe', 'Southeast Asia']


In [72]:
regional_yr = (df.groupby(['iyear', 'region_txt'])
                 .size()
                 .reset_index(name='attacks'))
 
# Preview: pivot table — year × top-3 regions
sample_pivot = (regional_yr[regional_yr['region_txt'].isin(TOP6[:3])]
                .pivot_table(index='iyear',
                             columns='region_txt',
                             values='attacks',
                             fill_value=0))
print("\nSample preview (top 3 regions, every 5th year):")
print(sample_pivot.iloc[::5].to_string())


Sample preview (top 3 regions, every 5th year):
region_txt  Middle East & North Africa  South Asia  Sub-Saharan Africa
iyear                                                                 
1970                              27.0         1.0                 3.0
1975                              44.0         4.0                12.0
1980                             419.0        12.0                52.0
1985                             131.0       152.0               139.0
1990                             485.0       555.0               302.0
1996                             368.0       585.0               198.0
2001                             361.0       385.0               161.0
2006                            1182.0       930.0               114.0
2011                            1636.0      2109.0               482.0
2016                            5342.0      3547.0              2011.0


In [73]:
g_series = global_trend.set_index('iyear')['attacks']
corr_rows = []
reg = region_totals['region_txt']
for region in reg:
    r_series = (regional_yr[regional_yr['region_txt'] == region]
                .set_index('iyear')['attacks'])
    aligned = pd.concat([r_series, g_series], axis=1, join='inner')
    aligned.columns = ['region', 'global']
    corr_rows.append({
        'Region'      : region,
        'Corr (r)'    : round(aligned['region'].corr(aligned['global']), 3),
        'Peak Year'   : int(r_series.idxmax()),
        'Peak Attacks': int(r_series.max()),
    })
 
corr_df = pd.DataFrame(corr_rows).sort_values('Corr (r)', ascending=False)
print(corr_df.to_string(index=False))
 
print("""
  Interpretation guide
  ────────────────────
  r ≥ 0.90  → closely follows global trend
  r ≈ 0.0   → independent / no relationship
  r < 0.0   → INVERSE — opposite of global trend
""")

                     Region  Corr (r)  Peak Year  Peak Attacks
 Middle East & North Africa     0.973       2014          6424
         Sub-Saharan Africa     0.959       2014          2273
                 South Asia     0.952       2014          4868
             Southeast Asia     0.949       2013          1152
             Eastern Europe     0.721       2014           914
      Australasia & Oceania     0.245       1989            29
                  East Asia     0.207       1990            87
              South America     0.001       1984          1259
             Western Europe    -0.108       1979           842
Central America & Caribbean    -0.141       1981          1077
               Central Asia    -0.224       1992            74
              North America    -0.232       1970           465

  Interpretation guide
  ────────────────────
  r ≥ 0.90  → closely follows global trend
  r ≈ 0.0   → independent / no relationship
  r < 0.0   → INVERSE — opposite of global tren

In [78]:
fig1, axes = plt.subplots(2, 2, figsize=(18, 11))
fig1.suptitle(
    'QUESTION 1  ─  HOW HAS TERRORIST ACTIVITY CHANGED?\n'
    'Global Trend  &  Regional Comparison  ·  1970–2017',
    fontsize=14, fontweight='bold', color='white', y=1.01)
 
yr  = global_trend['iyear'].values
atk = global_trend['attacks'].values
 
# ── A: Global area chart ─────────────────────────────────────
ax = axes[0, 0]
ax.fill_between(yr, atk, alpha=0.18, color=ACCENT)
ax.plot(yr, atk, color=ACCENT, linewidth=2.2)
ax.set_title('A  ·  Global Attack Frequency', color='white',
             fontsize=11, pad=8)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Attacks')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
 
# shade historical eras
for start, end, lbl in [
        (1970, 1993, 'COLD WAR'),
        (1993, 2001, 'POST-COLD\nWAR'),
        (2001, 2010, 'WAR ON\nTERROR'),
        (2010, 2017, 'MODERN\nERA')]:
    ax.axvspan(start, end, alpha=0.04, color='white')
    ax.text((start + end) / 2, atk.max() * 0.97, lbl,
            ha='center', fontsize=6, color='#555')
 
# resolve peak year row (works if peak_year is scalar year or row-like)
if np.isscalar(peak_year):
    peak_year_val = int(peak_year)
    peak_attack_val = int(global_trend.loc[
        global_trend['iyear'] == peak_year_val, 'attacks'].squeeze())
else:
    peak_year_val = int(peak_year['iyear'])
    peak_attack_val = int(peak_year['attacks'])

# annotate peak year
ax.annotate(
    f"PEAK {peak_year_val}\n{peak_attack_val:,} attacks",
    xy=(peak_year_val, peak_attack_val),
    xytext=(peak_year_val - 11, peak_attack_val * 0.80),
    fontsize=8, color=ACCENT,
    arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.2))
ax.grid(True, axis='y')
 
# ── B: Regional line chart ───────────────────────────────────
ax = axes[0, 1]
for region in TOP6:
    sub   = regional_yr[regional_yr['region_txt'] == region]
    short = (region.replace(' & North Africa', '')
                   .replace('Sub-Saharan ', 'SS '))
    ax.plot(sub['iyear'], sub['attacks'],
            label=short,
            color=REGION_COLORS[region],
            linewidth=1.9, alpha=0.92)
ax.set_title('B  ·  Regional Trends — Top 6 Regions',
             color='white', fontsize=11, pad=8)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Attacks')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=7.5, loc='upper left',
          facecolor='#1a1a1a', edgecolor='#333', labelcolor='#aaa')
ax.grid(True, axis='y')
 
# ── C: Stacked % share ───────────────────────────────────────
ax = axes[1, 0]
pct_data = (regional_yr[regional_yr['region_txt'].isin(TOP6)]
            .merge(global_trend.rename(columns={'attacks': 'global'}),
                   on='iyear')
            .assign(pct=lambda d: d['attacks'] / d['global'] * 100)
            .pivot_table(index='iyear', columns='region_txt',
                         values='pct', fill_value=0)
            .reindex(columns=TOP6, fill_value=0))
 
short_names = [(r.replace(' & North Africa', '')
                 .replace('Sub-Saharan ', 'SS '))
               for r in TOP6]
ax.stackplot(pct_data.index,
             [pct_data[r].values for r in TOP6],
             labels=short_names,
             colors=[REGION_COLORS[r] for r in TOP6],
             alpha=0.85)
ax.set_title('C  ·  Regional Share of Global Attacks (%)',
             color='white', fontsize=11, pad=8)
ax.set_xlabel('Year')
ax.set_ylabel('% of Global Attacks')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x)}%'))
ax.legend(fontsize=7, loc='upper left',
          facecolor='#1a1a1a', edgecolor='#333', labelcolor='#aaa')
ax.grid(True, axis='y')
 
# ── D: Decade grouped bar ────────────────────────────────────
ax = axes[1, 1]
decades = ['1970s', '1980s', '1990s', '2000s', '2010s']
x_pos   = np.arange(len(decades))
width   = 0.13
offset  = -(len(TOP6) - 1) / 2 * width

decade_by_region = (
    regional_yr
    .assign(decade=(regional_yr['iyear'] // 10 * 10).astype(int).astype(str) + 's')
    .groupby(['region_txt', 'decade'])['attacks']
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=decades, fill_value=0)
)

for i, region in enumerate(TOP6):
    vals  = decade_by_region.loc[region, decades].values
    short = (region.replace(' & North Africa', '')
                   .replace('Sub-Saharan ', 'SS '))
    ax.bar(x_pos + offset + i * width, vals,
           width=width * 0.9,
           color=REGION_COLORS[region],
           alpha=0.85, label=short)
 
ax.set_title('D  ·  Attacks per Decade — by Region',
             color='white', fontsize=11, pad=8)
ax.set_xlabel('Decade')
ax.set_ylabel('Total Attacks')
ax.set_xticks(x_pos)
ax.set_xticklabels(decades)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=7, loc='upper left',
          facecolor='#1a1a1a', edgecolor='#333', labelcolor='#aaa')
ax.grid(True, axis='y')
 
plt.tight_layout()
plt.savefig('q1_trend_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0e0e0e')
plt.close()
print("  ✔  Saved: q1_trend_analysis.png")

  ✔  Saved: q1_trend_analysis.png


q2 - korelasi attack and korban - irregulation or outlier

In [79]:
yearly = (df.groupby('iyear')
            .agg(incidents  = ('iyear', 'count'),
                 casualties = ('nkill', 'sum'))
            .reset_index())
 
print(yearly.to_string(index=False))

 iyear  incidents  casualties
  1970        643         174
  1971        460         173
  1972        532         566
  1973        471         370
  1974        572         539
  1975        727         616
  1976        888         674
  1977       1206         456
  1978       1448        1456
  1979       2377        2078
  1980       2485        4391
  1981       2479        4847
  1982       2410        5128
  1983       2719        9433
  1984       3079       10337
  1985       2435        6989
  1986       2436        4930
  1987       2599        6115
  1988       3423        7195
  1989       3951        8117
  1990       3391        6979
  1991       4078        8390
  1992       4637        9621
  1994       3174        7613
  1995       2521        5862
  1996       2537        6827
  1997       2725       10116
  1998        900        4653
  1999       1380        3390
  2000       1787        4403
  2001       1886        7722
  2002       1312        4796
  2003    

In [80]:
# pearson korelasi

r_val = yearly['incidents'].corr(yearly['casualties'])
print(f"\n  Pearson r  =  {r_val:.4f}")
print("""
  Strength guide
  ─────────────────────────────────
  0.90 – 1.00 → Very strong positive
  0.70 – 0.89 → Strong positive
  0.50 – 0.69 → Moderate positive
  < 0.50      → Weak
""")
 


  Pearson r  =  0.9650

  Strength guide
  ─────────────────────────────────
  0.90 – 1.00 → Very strong positive
  0.70 – 0.89 → Strong positive
  0.50 – 0.69 → Moderate positive
  < 0.50      → Weak



In [81]:
# detect outliers - kill per incident ratio

yearly['ratio'] = (yearly['casualties'] / yearly['incidents']).round(3)
 
mu     = yearly['ratio'].mean()
sigma  = yearly['ratio'].std()
thresh = mu + 1.5 * sigma
 
outliers = yearly[yearly['ratio'] > thresh].copy()
 
print(f"\n  Mean kill/incident   : {mu:.3f}")
print(f"  Std deviation        : {sigma:.3f}")
print(f"  Outlier threshold    : {thresh:.3f}  (mean + 1.5σ)")
print(f"\n  Flagged outlier years ({len(outliers)}):")
print(outliers[['iyear', 'incidents', 'casualties', 'ratio']]
      .to_string(index=False))



  Mean kill/incident   : 2.260
  Std deviation        : 1.128
  Outlier threshold    : 3.953  (mean + 1.5σ)

  Flagged outlier years (4):
 iyear  incidents  casualties  ratio
  1998        900        4653  5.170
  2001       1886        7722  4.094
  2004       1155        5741  4.971
  2007       3179       12769  4.017


In [84]:
# visualize correlation + outliers

fig2, axes2 = plt.subplots(1, 3, figsize=(20, 6))
fig2.suptitle(
    f'QUESTION 2  ─  INCIDENTS vs CASUALTIES  ·  '
    f'PEARSON r = {r_val:.3f}  ·  OUTLIER DETECTION',
    fontsize=13, fontweight='bold', color='white', y=1.01)
 
# ── A: Dual-axis time series ─────────────────────────────────
ax  = axes2[0]
ax2b = ax.twinx()
 
ax.fill_between(yearly['iyear'], yearly['incidents'],
                alpha=0.12, color=BLUE)
ax.plot(yearly['iyear'], yearly['incidents'],
        color=BLUE, linewidth=2, label='Incidents')
ax2b.plot(yearly['iyear'], yearly['casualties'],
          color=ACCENT, linewidth=2, linestyle='--', label='Casualties')
 
# style right-hand axis
ax2b.tick_params(colors='#555', labelsize=8)
ax2b.yaxis.label.set_color(ACCENT)
for sp in ax2b.spines.values():
    sp.set_color('#2c2c2c')
ax2b.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
 
ax.set_title('A  ·  Incidents & Casualties Over Time',
             color='white', fontsize=10, pad=8)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Incidents', color=BLUE)
ax2b.set_ylabel('Total Casualties', color=ACCENT)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
lines = ax.get_lines() + ax2b.get_lines()
ax.legend(lines, ['Incidents', 'Casualties'],
          facecolor='#1a1a1a', edgecolor='#333',
          labelcolor='#aaa', fontsize=8)
ax.grid(True, axis='y')
 
# ── B: Scatter + trend line ──────────────────────────────────
ax = axes2[1]
normal = yearly[yearly['ratio'] <= thresh]
 
ax.scatter(normal['incidents'], normal['casualties'],
           color=BLUE, alpha=0.75, s=55, zorder=3,
           label='Normal year')
ax.scatter(outliers['incidents'], outliers['casualties'],
           color=ACCENT, s=90, zorder=5,
           label=f'Outlier  (ratio > {thresh:.2f})')
 
# label each outlier point
for _, row in outliers.iterrows():
    ax.annotate(str(int(row['iyear'])),
                xy=(row['incidents'], row['casualties']),
                xytext=(9, 4), textcoords='offset points',
                fontsize=8, color=ACCENT)
 
# OLS trend line
z  = np.polyfit(yearly['incidents'], yearly['casualties'], 1)
xs = np.linspace(yearly['incidents'].min(),
                 yearly['incidents'].max(), 300)
ax.plot(xs, np.poly1d(z)(xs),
        color=ORANGE, linewidth=1.5, linestyle='--',
        alpha=0.85, label='Linear trend')
 
ax.set_title(f'B  ·  Correlation Scatter  (r = {r_val:.3f})',
             color='white', fontsize=10, pad=8)
ax.set_xlabel('Number of Incidents')
ax.set_ylabel('Total Casualties')
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(facecolor='#1a1a1a', edgecolor='#333',
          labelcolor='#aaa', fontsize=8)
ax.grid(True)
 
# ── C: Ratio bar chart ───────────────────────────────────────
ax = axes2[2]
bar_cols = [ACCENT if v > thresh else BLUE
            for v in yearly['ratio']]
ax.bar(yearly['iyear'], yearly['ratio'],
       color=bar_cols, width=0.85, alpha=0.85)
 
ax.axhline(mu,     color=ORANGE, linewidth=1.4, linestyle='--',
           label=f'Mean ({mu:.2f})')
ax.axhline(thresh, color=ACCENT,    linewidth=1.2, linestyle=':',
           label=f'Threshold ({thresh:.2f})')
 
# label each flagged bar
for _, row in outliers.iterrows():
    ax.text(row['iyear'], row['ratio'] + 0.06,
            str(int(row['iyear'])),
            ha='center', fontsize=6.5, color=ACCENT)
 
ax.set_title('C  ·  Kill-per-Incident Ratio by Year',
             color='white', fontsize=10, pad=8)
ax.set_xlabel('Year')
ax.set_ylabel('Avg Deaths per Incident')
ax.legend(facecolor='#1a1a1a', edgecolor='#333',
          labelcolor='#aaa', fontsize=8)
ax.grid(True, axis='y')
 
plt.tight_layout()
plt.savefig('q2_correlation_outliers.png', dpi=150,
            bbox_inches='tight', facecolor='#0e0e0e')
plt.close()
print("  ✔  Saved: q2_correlation_outliers.png")

  ✔  Saved: q2_correlation_outliers.png


In [88]:
# attack method distribution
method_global = (df['attacktype1_txt']
                 .value_counts()
                 .reset_index())
method_global.columns = ['method', 'count']
method_global['pct'] = (method_global['count'] /
                         method_global['count'].sum() * 100).round(2)
print(method_global.to_string(index=False))

                             method  count   pct
                  Bombing/Explosion  80879 47.46
                      Armed Assault  41146 24.14
                      Assassination  18886 11.08
        Hostage Taking (Kidnapping)  10946  6.42
     Facility/Infrastructure Attack   9198  5.40
                            Unknown   6856  4.02
Hostage Taking (Barricade Incident)    942  0.55
                    Unarmed Assault    928  0.54
                          Hijacking    649  0.38


In [92]:
# attack methods regionally & by decade

method_region = (df.groupby(['region_txt', 'attacktype1_txt'])
                   .size()
                   .reset_index(name='count'))
 
# top-1 method per region
top_per_region = (method_region
                  .sort_values('count', ascending=False)
                  .groupby('region_txt')
                  .first()
                  .reset_index()
                  [['region_txt', 'attacktype1_txt', 'count']])
print("\nDominant attack method per region:")
print(top_per_region.to_string(index=False))
 
# % share matrix (top-5 methods × all regions)
top5m = method_global['method'].head(5).tolist()
pivot = (method_region
         .pivot_table(index='region_txt',
                      columns='attacktype1_txt',
                      values='count',
                      fill_value=0)
         .reindex(columns=top5m, fill_value=0))
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
 
print("\nRegion × method — row-normalised % share (top 5 methods):")
print(pivot_pct.round(1).to_string())


Dominant attack method per region:
                 region_txt   attacktype1_txt  count
      Australasia & Oceania Bombing/Explosion     75
Central America & Caribbean     Armed Assault   4199
               Central Asia Bombing/Explosion    225
                  East Asia Bombing/Explosion    272
             Eastern Europe Bombing/Explosion   2657
 Middle East & North Africa Bombing/Explosion  28730
              North America Bombing/Explosion   1476
              South America Bombing/Explosion   7294
                 South Asia Bombing/Explosion  20587
             Southeast Asia Bombing/Explosion   4557
         Sub-Saharan Africa     Armed Assault   5870
             Western Europe Bombing/Explosion   7484

Region × method — row-normalised % share (top 5 methods):
attacktype1_txt              Bombing/Explosion  Armed Assault  Assassination  Hostage Taking (Kidnapping)  Facility/Infrastructure Attack
region_txt                                                                    

In [94]:
# attack method over time

method_decade = (df.groupby(['decade', 'attacktype1_txt'])
                   .size()
                   .reset_index(name='count')
                   .pivot_table(index='attacktype1_txt',
                                columns='decade',
                                values='count',
                                fill_value=0)
                   .reindex(columns=['1970s','1980s','1990s','2000s','2010s'],
                            fill_value=0))
method_decade['TOTAL'] = method_decade.sum(axis=1)
method_decade = method_decade.sort_values('TOTAL', ascending=False)
print(method_decade.to_string())

decade                                1970s    1980s   1990s    2000s    2010s    TOTAL
attacktype1_txt                                                                        
Bombing/Explosion                    3950.0  11742.0  9204.0  13227.0  42756.0  80879.0
Armed Assault                        1371.0   7394.0  6434.0   6286.0  19661.0  41146.0
Assassination                        1959.0   5095.0  4893.0   1446.0   5493.0  18886.0
Hostage Taking (Kidnapping)           536.0   1104.0  1417.0   1682.0   6207.0  10946.0
Facility/Infrastructure Attack       1011.0   1330.0  1575.0   1289.0   3993.0   9198.0
Unknown                               255.0    896.0  1270.0    516.0   3919.0   6856.0
Hostage Taking (Barricade Incident)   156.0    302.0   125.0     37.0    322.0    942.0
Unarmed Assault                        25.0     44.0   256.0    178.0    425.0    928.0
Hijacking                              61.0    109.0   169.0     81.0    229.0    649.0


In [95]:
# visualize

fig3, axes3 = plt.subplots(1, 3, figsize=(21, 7))
fig3.suptitle(
    'QUESTION 3  ─  MOST COMMON ATTACK METHODS\n'
    'Global Distribution  ·  By Region  ·  Over Time',
    fontsize=13, fontweight='bold', color='white', y=1.01)
 
# ── A: Horizontal bar ────────────────────────────────────────
ax = axes3[0]
mg      = method_global.copy()
bcolors = [METHOD_COLORS.get(m, '#555') for m in mg['method']]
bars    = ax.barh(mg['method'], mg['count'],
                  color=bcolors, height=0.65)
 
for bar, pct in zip(bars, mg['pct']):
    ax.text(bar.get_width() + 300,
            bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=8.5, color='#aaa')
 
ax.set_title('A  ·  Global Distribution',
             color='white', fontsize=10, pad=8)
ax.set_xlabel('Number of Attacks')
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.invert_yaxis()
ax.set_xlim(0, mg['count'].max() * 1.22)
ax.grid(True, axis='x')
 
# ── B: Heatmap — region × method ─────────────────────────────
ax = axes3[1]
top8_regions = (df['region_txt'].value_counts().head(8).index.tolist())
top6m        = method_global['method'].head(6).tolist()
 
hm = (method_region[
        method_region['region_txt'].isin(top8_regions) &
        method_region['attacktype1_txt'].isin(top6m)]
      .pivot_table(index='region_txt', columns='attacktype1_txt',
                   values='count', fill_value=0)
      .reindex(columns=top6m, fill_value=0))
hm_pct = hm.div(hm.sum(axis=1), axis=0) * 100
 
im = ax.imshow(hm_pct.values, cmap='YlOrRd',
               aspect='auto', vmin=0, vmax=75)
 
# x-axis: shortened method names
short_m = [m.replace('Hostage Taking (Kidnapping)', 'HT Kidnap')
            .replace('Facility/Infrastructure Attack', 'Facility Atk')
            .replace('Hostage Taking (Barricade Incident)', 'HT Barricade')
            for m in top6m]
ax.set_xticks(range(len(top6m)))
ax.set_xticklabels(short_m, rotation=28, ha='right',
                   fontsize=7.5, color='#aaa')
 
# y-axis: shortened region names
short_r = [(r.replace(' & North Africa', '')
             .replace('Sub-Saharan ', 'SS ')
             .replace('Central America & Caribbean', 'C.Am & Carib.'))
           for r in hm.index]
ax.set_yticks(range(len(hm.index)))
ax.set_yticklabels(short_r, fontsize=8, color='#aaa')
 
# annotate cells with % values
for i in range(hm_pct.shape[0]):
    for j in range(hm_pct.shape[1]):
        v = hm_pct.values[i, j]
        ax.text(j, i, f'{v:.0f}%', ha='center', va='center',
                fontsize=7, color='white' if v > 45 else '#222')
 
ax.set_title('B  ·  % Share by Method per Region\n(row-normalised)',
             color='white', fontsize=10, pad=8)
plt.colorbar(im, ax=ax, shrink=0.72, label='% share')
 
# ── C: Stacked area — methods over time ──────────────────────
ax = axes3[2]
method_yr = (df.groupby(['iyear', 'attacktype1_txt'])
               .size()
               .reset_index(name='count')
               .pivot_table(index='iyear', columns='attacktype1_txt',
                            values='count', fill_value=0)
               .reindex(columns=top5m, fill_value=0))
 
ax.stackplot(
    method_yr.index,
    [method_yr[m].values for m in top5m],
    labels=[m.replace('Hostage Taking (Kidnapping)', 'HT Kidnap')
             .replace('Facility/Infrastructure Attack', 'Facility Atk')
             for m in top5m],
    colors=[METHOD_COLORS.get(m, '#555') for m in top5m],
    alpha=0.85)
 
ax.set_title('C  ·  Attack Methods Over Time (Top 5)',
             color='white', fontsize=10, pad=8)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Attacks')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=7.5, loc='upper left',
          facecolor='#1a1a1a', edgecolor='#333', labelcolor='#aaa')
ax.grid(True, axis='y')
 
plt.tight_layout()
plt.savefig('q3_attack_methods.png', dpi=150,
            bbox_inches='tight', facecolor='#0e0e0e')
plt.close()
print("  ✔  Saved: q3_attack_methods.png")

  ✔  Saved: q3_attack_methods.png


visualisasi region yang terserang

In [104]:
# prepare data georgrafi
import bz2
import pandas as pd

# decompress and load in one step — no temp file needed
with bz2.open('globalterrorismdb_0718dist.tar.bz2', 'rt', encoding='utf-8') as f:
    df = pd.read_csv(f, low_memory=False)

map_df = (df.dropna(subset=['latitude', 'longitude'])
           .query('-90 <= latitude <= 90 and -180 <= longitude <= 180')
           .copy())

print(f"Valid coordinates : {len(map_df):,} rows "
      f"({len(map_df)/len(df)*100:.1f}% of dataset)")

country = (map_df.groupby(['country_txt', 'region_txt'])
                  .agg(total_attacks = ('iyear',  'count'),
                       total_killed  = ('nkill',  'sum'),
                       lat           = ('latitude',  'mean'),
                       lon           = ('longitude', 'mean'))
                  .reset_index())

print(f"Unique countries  : {len(country)}")
print("\nTop 12 countries by total attacks:")
print(country.sort_values('total_attacks', ascending=False)
             .head(12)
             [['country_txt', 'total_attacks', 'total_killed']]
             .to_string(index=False))

Valid coordinates : 177,133 rows (97.5% of dataset)
Unique countries  : 204

Top 12 countries by total attacks:
   country_txt  total_attacks  total_killed
          Iraq          24487       78175.0
      Pakistan          14318       23679.0
   Afghanistan          12639       39247.0
         India          11801       18663.0
      Colombia           7835       13514.0
   Philippines           6528        7969.0
          Peru           5808       11306.0
United Kingdom           5227        3410.0
   El Salvador           4846       11322.0
        Turkey           4126        6087.0
       Somalia           4120       10190.0
       Nigeria           3866       22659.0


In [105]:
# sample for speed — random 30 000 points
sample = map_df.sample(n=min(30_000, len(map_df)), random_state=42)
 
fig4, (ax_map, ax_bar) = plt.subplots(
    2, 1, figsize=(18, 14),
    gridspec_kw={'height_ratios': [3, 1]})
fig4.suptitle(
    'QUESTION 4  ─  GEOGRAPHIC SPREAD OF TERRORIST ATTACKS  ·  1970–2017\n'
    'Bubble size  ∝  total attacks per country',
    fontsize=13, fontweight='bold', color='white', y=0.98)
 
# ── MAP PANEL ────────────────────────────────────────────────
ax_map.set_facecolor('#091524')
ax_map.set_xlim(-180, 180)
ax_map.set_ylim(-60,   85)
 
# graticule lines
for lon in range(-180, 181, 30):
    ax_map.axvline(lon, color='#122236', linewidth=0.3, zorder=1)
for lat in range(-60, 86, 20):
    ax_map.axhline(lat, color='#122236', linewidth=0.3, zorder=1)
ax_map.axhline(0, color='#1a3350', linewidth=0.7, zorder=1)  # equator
 
# raw incident scatter — one dot per sampled attack
for region, color in REGION_COLORS.items():
    sub = sample[sample['region_txt'] == region]
    ax_map.scatter(sub['longitude'], sub['latitude'],
                   c=color, s=1.0, alpha=0.20,
                   linewidths=0, zorder=2)
 
# country-level bubble overlay
c_max = country['total_attacks'].max()
for _, row in country.iterrows():
    sz    = (row['total_attacks'] / c_max) * 700 + 8
    color = REGION_COLORS.get(row['region_txt'], '#888')
    ax_map.scatter(row['lon'], row['lat'],
                   s=sz, c=color, alpha=0.55,
                   edgecolors='white', linewidths=0.3, zorder=3)
 
# label top-10 hotspot countries
top10 = country.sort_values('total_attacks', ascending=False).head(10)
for _, row in top10.iterrows():
    ax_map.annotate(
        row['country_txt'],
        xy=(row['lon'], row['lat']),
        xytext=(4, 4), textcoords='offset points',
        fontsize=6.5, color='white', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2',
                  fc='#000000bb', ec='none'))
 
# ── legend: regions ──────────────────────────────────────────
region_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=c, markersize=6,
           label=(r.replace(' & North Africa', '')
                   .replace('Sub-Saharan ', 'SS ')))
    for r, c in REGION_COLORS.items()
]
# legend: bubble size guide
size_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#999', markersize=ms,
           alpha=0.6, label=lbl)
    for ms, lbl in [(3, '< 500'), (7, '~5,000'), (11, '>15,000')]
]
ax_map.legend(
    handles=region_handles + size_handles,
    ncol=2, fontsize=6.5, loc='lower left',
    facecolor='#0d1f30', edgecolor='#2a3a4a',
    labelcolor='#cccccc',
    title='Region  (bubble size = attacks)',
    title_fontsize=6.5)
 
ax_map.set_xlabel('Longitude', fontsize=8)
ax_map.set_ylabel('Latitude',  fontsize=8)
ax_map.tick_params(colors='#445', labelsize=7)
ax_map.set_title(
    'Attack locations — 30k sample points + country bubble overlay',
    color='white', fontsize=10, pad=8)
 
# ── BAR PANEL ────────────────────────────────────────────────
reg_total = (df.groupby('region_txt')
               .size()
               .sort_values(ascending=True))
bar_colors = [REGION_COLORS.get(r, '#888') for r in reg_total.index]
ax_bar.barh(reg_total.index, reg_total.values,
            color=bar_colors, height=0.65)
 
for i, (region, val) in enumerate(reg_total.items()):
    ax_bar.text(val + 200, i, f'{val:,}',
                va='center', fontsize=7.5, color='#aaa')
 
ax_bar.set_title('Total Attacks by Region  ·  1970–2017',
                 color='white', fontsize=10, pad=8)
ax_bar.set_xlabel('Total Attacks')
ax_bar.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax_bar.set_xlim(0, reg_total.max() * 1.14)
ax_bar.grid(True, axis='x')
 
plt.tight_layout()
plt.savefig('q4_world_map.png', dpi=150,
            bbox_inches='tight', facecolor='#0e0e0e')
plt.close()
print("  ✔  Saved: q4_world_map.png")

  ✔  Saved: q4_world_map.png


# conclusion

In [111]:
peak_year    = global_trend.loc[global_trend['attacks'].idxmax()]
min_year = global_trend.loc[global_trend['attacks'].idxmin()]

print(f"Peak     : {int(peak_year['iyear'])}  →  {int(peak_year['attacks']):,} attacks")
print(f"Lowest   : {int(min_year['iyear'])}  →  {int(min_year['attacks']):,} attacks")
print(f"Mean/year: {global_trend['attacks'].mean():,.0f} attacks")

Peak     : 2014  →  16,109 attacks
Lowest   : 1971  →  460 attacks
Mean/year: 3,626 attacks


In [ ]:
top_country = country.sort_values('total_attacks', ascending=False).iloc[0]
corr_lookup = corr_df.set_index('Region')
 
print(f"""
Q1  TREND OVER YEARS & REGIONAL DIFFERENCES
──────────────────────────────────────────────────────────────
• Attacks rose from {int(min_year['attacks']):,}/yr ({int(min_year['iyear'])})
  to a peak of {int(peak_year['attacks']):,} in {int(peak_year['iyear'])} — a 25× increase.
• Biggest decade jump: 2010s  (+236% vs 2000s)
• Regions that FOLLOW global trend (r ≥ 0.90):
    Middle East & N. Africa  r = {corr_lookup.loc['Middle East & North Africa','Corr (r)']}
    Sub-Saharan Africa        r = {corr_lookup.loc['Sub-Saharan Africa','Corr (r)']}
    South Asia                r = {corr_lookup.loc['South Asia','Corr (r)']}
    Southeast Asia            r = {corr_lookup.loc['Southeast Asia','Corr (r)']}
• Regions that DIVERGE (own pattern):
    South America  r ≈ {corr_lookup.loc['South America','Corr (r)']}  — peaked 1980s (FARC/Shining Path), then fell
    Western Europe r ≈ {corr_lookup.loc['Western Europe','Corr (r)']}  — peaked 1970s (IRA/ETA), then steadily declined
 
Q2  INCIDENTS vs CASUALTIES CORRELATION
──────────────────────────────────────────────────────────────
• Pearson r = {r_val:.3f}  → very strong positive correlation
• Both series rise together almost perfectly
• Outlier years: {', '.join(outliers['iyear'].astype(str).tolist())}
  → Kill/incident ratio > {thresh:.2f}  (mean + 1.5σ)
  → Caused by mass-casualty events that inflated deaths
    relative to the number of attacks that year
 
Q3  MOST COMMON ATTACK METHODS
──────────────────────────────────────────────────────────────
• #1  Bombing/Explosion    — {method_global.iloc[0]['pct']:.1f}% of all attacks
• #2  Armed Assault        — {method_global.iloc[1]['pct']:.1f}%
• #3  Assassination        — {method_global.iloc[2]['pct']:.1f}%
• Bombing dominates every region EXCEPT:
    Sub-Saharan Africa     → Armed Assault is #1
    Central America        → Armed Assault is #1
• Over time: Bombing's share exploded in the 2010s
  as ISIS/AQ-linked groups ramped up IED campaigns
 
Q4  GEOGRAPHIC SPREAD
──────────────────────────────────────────────────────────────
• Hottest country  : {top_country['country_txt']}  ({top_country['total_attacks']:,} attacks)
• Top-3 clusters   : Iraq/Syria · Afghanistan/Pakistan · India
• Quietest zones   : Oceania, Northern Europe, North America
• MENA + South Asia: >50% of all recorded incidents
""")
 
print("=" * 65)
print("✅  Analysis complete.  Four figures saved:")
print("     q1_trend_analysis.png")
print("     q2_correlation_outliers.png")
print("     q3_attack_methods.png")
print("     q4_world_map.png")
print("=" * 65)
 


Q1  TREND OVER YEARS & REGIONAL DIFFERENCES
──────────────────────────────────────────────────────────────
• Attacks rose from 460/yr (1971)
  to a peak of 16,109 in 2014 — a 25× increase.
• Biggest decade jump: 2010s  (+236% vs 2000s)
• Regions that FOLLOW global trend (r ≥ 0.90):
    Middle East & N. Africa  r = 0.973
    Sub-Saharan Africa        r = 0.959
    South Asia                r = 0.952
    Southeast Asia            r = 0.949
• Regions that DIVERGE (own pattern):
    South America  r ≈ 0.001  — peaked 1980s (FARC/Shining Path), then fell
    Western Europe r ≈ -0.108  — peaked 1970s (IRA/ETA), then steadily declined

Q2  INCIDENTS vs CASUALTIES CORRELATION
──────────────────────────────────────────────────────────────
• Pearson r = 0.965  → very strong positive correlation
• Both series rise together almost perfectly
• Outlier years: 1998, 2001, 2004, 2007
  → Kill/incident ratio > 3.95  (mean + 1.5σ)
  → Caused by mass-casualty events that inflated deaths
    relative to 

: 